# Accessible Design Review Panel

A UX designer pastes a component description or design spec. Three LLM reviewers each take a different perspective: one reviews from a screen reader user's perspective, one from a keyboard-only user's perspective, and one from a cognitive disability perspective. A fourth LLM synthesizes the three reviews into a prioritized list of design changes with WCAG criterion references. Useful for design teams who want accessibility feedback before a component is built.

Sessions can be saved to disk and analyzed over time: a built-in trend pipeline tallies the most frequently flagged WCAG criteria across all saved sessions and asks Claude to produce a plain-language training briefing — helping teams spot systemic gaps in their design practice and prioritize what to learn next.

## Reviewer Personas

The panel consists of three specialized reviewers followed by a synthesizer:

**Reviewer 1 — Screen Reader User**
Reviews the component as a blind user who navigates entirely by screen reader (JAWS, NVDA, VoiceOver). Focuses on: semantic HTML and ARIA role correctness, meaningful accessible name and description for all interactive elements, logical reading order, live region announcements for dynamic content, and whether the component is perceivable and understandable without visual context.

**Reviewer 2 — Keyboard-Only User**
Reviews the component as a user with motor impairments who navigates entirely by keyboard, no mouse. Focuses on: tab order and focus management, visible focus indicators, keyboard interaction patterns (matching the ARIA Authoring Practices Guide), focus trap avoidance, and whether all functionality is reachable and operable without a pointing device.

**Reviewer 3 — Cognitive Disability Perspective**
Reviews the component as a user with cognitive, learning, or attention disabilities. Focuses on: plain language labels and instructions, consistent and predictable behavior, error identification and recovery, time limits, distraction-free layout, and whether the component is understandable and forgiving for users who process information differently.

**Synthesizer — Accessibility Panel Chair**
Receives all three reviews and the original component spec and produces a single prioritized list of design changes, each tagged with the relevant WCAG success criterion, conformance level (A / AA / AAA), and an implementation priority (critical / recommended / enhancement).

In [ ]:
# imports

import os
import json
import time
import glob
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPEN_ROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini and Anthropic we use the OpenAI python client
# because both providers expose an OpenAI-compatible endpoint

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

## Step 1: Reviewer personas and session log

Defines `REVIEWER_PERSONAS`, a dict mapping short persona keys (`screen_reader`, `keyboard`, `cognitive`) to human-readable labels, and initializes an empty `conversation_log` list that will accumulate session records.

In [4]:
# Constants
REVIEWER_PERSONAS = {
    "screen_reader": "Screen Reader User",
    "keyboard":      "Keyboard-Only User",
    "cognitive":     "Cognitive Disability Perspective",
}

conversation_log = []

## Step 2: Reviewer system prompts and dispatch function

Defines three persona-specific system prompts (screen reader, keyboard-only, cognitive) and a `REVIEWER_CONFIG` dict that maps each persona key to its prompt, client, and model. The `get_review(component_spec, persona)` function dispatches to the correct client and returns the review text, with retry logic for rate limit errors.

In [ ]:
screen_reader_system_prompt = """You are an experienced blind web user who relies entirely on a screen reader (JAWS, NVDA, or VoiceOver) to use websites and applications. You have no access to visual layout, color, icons, or spatial positioning — everything you know about an interface comes from what your screen reader announces.

You are reviewing a UI component description or design spec provided by a UX designer, BEFORE it gets built. Your job is to identify accessibility issues that would affect someone navigating this component by screen reader.

For each issue you find:
- Describe the specific problem in concrete terms (e.g. "The icon-only 'X' button has no accessible name, so VoiceOver will announce it as just 'button'" — not "this may be unclear").
- Cite the relevant WCAG success criterion and conformance level, e.g. "WCAG 4.1.2 Name, Role, Value (Level A)".
- Explain briefly what a screen reader user would actually hear or experience as a result.
- Suggest a specific, concrete fix (exact ARIA attribute, label text, role, or markup change).

Focus areas to check:
- Semantic HTML and correct ARIA roles (prefer native elements over ARIA where possible)
- Accessible names and descriptions for every interactive element (aria-label, aria-labelledby, aria-describedby)
- Landmark regions and heading structure (logical h1-h6 hierarchy, no skipped levels)
- Logical reading order — does the DOM order match the order a screen reader user needs to understand the component, regardless of visual position?
- ARIA live regions for dynamic content (status messages, validation errors, loading states, content that appears/disappears) — will the user be told when something changes?
- Image alt text and decorative image handling
- Form labels, instructions, and error messages programmatically associated with their fields
- Descriptive link and button text (avoid "click here" / "read more" with no context)

End with a short "What's working well" section — call out anything in the spec that is already accessible or shows good practice. Be specific and constructive; the goal is to help the team ship something usable, not to overwhelm them."""

keyboard_system_prompt = """You are a web user with a motor impairment that prevents you from using a mouse, trackpad, or touchscreen. You operate every website and application entirely with a keyboard — Tab, Shift+Tab, Enter, Space, Escape, and arrow keys are your only tools.

You are reviewing a UI component description or design spec provided by a UX designer, BEFORE it gets built. Your job is to identify accessibility issues that would affect someone navigating and operating this component with a keyboard only.

For each issue you find:
- Describe the specific problem in concrete terms (e.g. "The spec doesn't say how the modal can be closed with the keyboard, and doesn't mention moving focus into it on open" — not "keyboard support may need work").
- Cite the relevant WCAG success criterion and conformance level, e.g. "WCAG 2.1.2 No Keyboard Trap (Level A)".
- Explain what would go wrong for a keyboard user as a result.
- Suggest a specific fix (where focus should move, which key triggers which action, which ARIA pattern to follow).

Focus areas to check:
- Tab order — does it follow a logical, predictable sequence (WCAG 2.4.3 Focus Order)?
- Visible focus indicators on every interactive element (WCAG 2.4.7 Focus Visible) — never rely on removing the default outline without a visible replacement
- All functionality operable via keyboard, with no mouse-only or hover-only interactions (WCAG 2.1.1 Keyboard)
- No keyboard traps — focus must always be able to move away from the component (WCAG 2.1.2 No Keyboard Trap)
- Correct interaction patterns per the WAI-ARIA Authoring Practices Guide (e.g. roving tabindex and arrow-key navigation for composite widgets like menus, tabs, listboxes, and comboboxes)
- Focus management on open/close of dialogs, menus, and dynamic content — focus moves into the new content, and returns to the triggering element on close
- Skip links / bypass blocks for repeated navigation (WCAG 2.4.1 Bypass Blocks)
- Conflicts between custom keyboard shortcuts and assistive technology or browser shortcuts (WCAG 2.1.4 Character Key Shortcuts)

End with a short "What's working well" section — call out anything in the spec that already supports good keyboard interaction. Be specific and actionable enough that a developer could implement your fixes directly."""

cognitive_system_prompt = """You are a web user with a cognitive, learning, or attention-related disability (for example, dyslexia, ADHD, or a memory-related condition). Dense text, unclear instructions, inconsistent navigation, time pressure, and busy or moving interfaces make it significantly harder for you to use a website successfully.

You are reviewing a UI component description or design spec provided by a UX designer, BEFORE it gets built. Your job is to identify accessibility issues that would affect someone with cognitive, learning, or attention disabilities using this component.

For each issue you find:
- Describe the specific problem in concrete terms (e.g. "The error message 'Invalid input' doesn't say what was wrong or how to fix it" — not "error handling could be improved").
- Cite the relevant WCAG success criterion and conformance level, e.g. "WCAG 3.3.3 Error Suggestion (Level AA)".
- Explain what would go wrong for someone with a cognitive disability as a result.
- Suggest a specific fix (revised wording, a design adjustment, or an added control).

Focus areas to check:
- Plain language in labels, instructions, and error messages — short sentences, common words, no unexplained jargon (WCAG 3.1 Readable)
- Consistent identification of components and predictable navigation/behavior across the interface (WCAG 3.2.3 Consistent Navigation, 3.2.4 Consistent Identification)
- Clear error identification, with messages that explain what went wrong and how to fix it (WCAG 3.3.1 Error Identification, 3.3.3 Error Suggestion)
- Time limits — are they adjustable, extendable, or removable, and is the user warned before time runs out (WCAG 2.2.1 Timing Adjustable)?
- Motion, animation, and auto-playing content — can it be paused, stopped, or hidden, and does it avoid triggering attention or vestibular issues (WCAG 2.3.3 Animation from Interactions)?
- Cognitive load — how much does the user need to remember, track, or hold in working memory to complete the task? Can steps be reduced or broken up?
- Distraction — are there competing elements (animations, auto-advancing carousels, pop-ups) that pull attention away from the primary task?
- Confirmation and recoverability — for destructive or hard-to-reverse actions, is there a confirmation step or an undo (WCAG 3.3.4 Error Prevention)?

End with a short "What's working well" section — call out anything in the spec that already reduces cognitive load or supports understanding. Be specific and constructive."""


# Shared by get_review() and ask_followup() so follow-up questions use the
# same system prompt, client, and model as the reviewer's original pass.
REVIEWER_CONFIG = {
    "screen_reader": (screen_reader_system_prompt, anthropic, "claude-sonnet-4-6"),
    "keyboard":      (keyboard_system_prompt,      openai,    "gpt-4o-mini"),
    "cognitive":     (cognitive_system_prompt,     gemini,    "gemini-2.5-flash"),
}


def get_review(component_spec, persona, max_retries=3):
    system_prompt, client, model = REVIEWER_CONFIG[persona]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": component_spec},
    ]
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model=model, messages=messages)
            return response.choices[0].message.content
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 20 * (attempt + 1)
            display(Markdown(f"*Rate limited on the {persona} reviewer — waiting {wait_seconds}s before retrying...*"))
            time.sleep(wait_seconds)


## Step 3: Collect the component description

`prompt_for_component()` displays guidance on what makes a useful component description, collects the designer's input via `input()`, and returns it as a string. The result is stored in `component_spec`.

In [9]:
example_input = """A modal dialog for confirming a file deletion. It has an <h2> heading
'Delete file?', body text 'This action cannot be undone.', a red 'Delete'
button, and a 'Cancel' link. Clicking Delete closes the modal and deletes
the file. Pressing Escape should also close the modal. The modal appears
after the user clicks a trash icon on a file card."""


def prompt_for_component():
    display(Markdown(f"""
## Describe your component

Paste a description of the UI component you want reviewed. Include as much
detail as possible:

- **Component type** — e.g. modal dialog, dropdown menu, date picker, data table
- **User interaction** — how a user opens, operates, and closes it
- **Labels and text** — button labels, field labels, placeholder text, error messages
- **HTML/ARIA markup** — paste any existing code if you have it
- **Dynamic behavior** — what changes on screen when the user interacts

**Example you can imitate:**

```
{example_input}
```
"""))
    spec = input("Paste your component description here: ").strip()
    while not spec:
        spec = input("Component description cannot be empty — please paste a description: ").strip()
    return spec


component_spec = prompt_for_component()


## Describe your component

Paste a description of the UI component you want reviewed. Include as much
detail as possible:

- **Component type** — e.g. modal dialog, dropdown menu, date picker, data table
- **User interaction** — how a user opens, operates, and closes it
- **Labels and text** — button labels, field labels, placeholder text, error messages
- **HTML/ARIA markup** — paste any existing code if you have it
- **Dynamic behavior** — what changes on screen when the user interacts

**Example you can imitate:**

```
A modal dialog for confirming a file deletion. It has an <h2> heading
'Delete file?', body text 'This action cannot be undone.', a red 'Delete'
button, and a 'Cancel' link. Clicking Delete closes the modal and deletes
the file. Pressing Escape should also close the modal. The modal appears
after the user clicks a trash icon on a file card.
```


## Step 4: Run all three reviewers

`run_reviewers(component_spec)` loops over each persona, calls `get_review()`, and renders each response as Markdown immediately after it arrives. Returns a dict of all three review texts keyed by persona.

In [10]:
def run_reviewers(component_spec):
    reviews = {}
    for key, label in REVIEWER_PERSONAS.items():
        display(Markdown(f"---\n## Reviewing as: {label}\n*Calling model...*"))
        review_text = get_review(component_spec, key)
        display(Markdown(f"---\n## Review: {label}\n\n{review_text}"))
        reviews[key] = review_text
    return reviews

reviews = run_reviewers(component_spec)


---
## Reviewing as: Screen Reader User
*Calling model...*

---
## Review: Screen Reader User

# Accessibility Review: Newsletter Subscription Popup

---

## Issue 1: Modal Not Implemented as a True Dialog

**Problem:** The popup is built as a plain `<div class="modal-popup">` with no ARIA role, no `aria-modal` attribute, and no `aria-labelledby`. A screen reader will not recognize this as a modal dialog — it will treat it as ordinary page content and announce nothing special when it appears.

**WCAG:** 4.1.2 Name, Role, Value (Level A)

**What the user hears:** Nothing. The popup silently appears over the page. A JAWS or NVDA user tabbing through page content will either never encounter the popup at all, or stumble into it mid-flow with no context about what it is. VoiceOver users will hear no announcement that an overlay has appeared.

**Fix:** Replace the outer `<div>` with a native `<dialog>` element, or at minimum apply `role="dialog"`, `aria-modal="true"`, and `aria-labelledby` pointing to the heading:
```html
<div role="dialog"
     aria-modal="true"
     aria-labelledby="newsletter-heading">
```

---

## Issue 2: Focus Is Not Moved Into the Popup on Open

**Problem:** When the popup appears, keyboard focus remains wherever it was on the background page. Tab navigation does not enter the popup, meaning a keyboard or screen reader user may never reach any of the popup's interactive elements.

**WCAG:** 2.1.1 Keyboard (Level A); also 2.1.2 No Keyboard Trap (Level A) — though here the problem is the inverse: the user cannot get *in*.

**What the user hears:** Nothing changes. The user continues tabbing through the background page, completely unaware a popup exists, and cannot interact with it.

**Fix:** On popup open, programmatically move focus to the first focusable element inside the dialog (typically the heading or the email input), and implement a **focus trap** so Tab and Shift+Tab cycle only within the popup while it is open:
```javascript
// On open:
document.querySelector('#newsletter-heading').focus();
// Trap focus within dialog using keydown listener
```

---

## Issue 3: Close Button Has No Accessible Name and Is Not Keyboard Operable

**Problem:** The close control is `<span>×</span>` — a `<span>` is not interactive, so it receives no keyboard focus, has no role, and the `×` character (a typographic multiplication sign) will be announced by most screen readers as "times" or "multiplication sign," not as "close."

**WCAG:** 4.1.2 Name, Role, Value (Level A); 2.1.1 Keyboard (Level A)

**What the user hears:** Either nothing (if the span is never reached via Tab), or "times" or "multiplication sign" with no indication it closes anything. Even if the user discovers it, pressing Enter or Space will do nothing.

**Fix:** Replace with a native `<button>` element (which is keyboard-operable by default) with a descriptive `aria-label`:
```html
<button type="button"
        aria-label="Close newsletter signup">
  <span aria-hidden="true">×</span>
</button>
```

---

## Issue 4: Escape Key Does Not Close the Popup

**Problem:** The spec states pressing Escape does nothing. Escape-to-dismiss is a firmly established and expected behavior for modal dialogs, and is explicitly required by the ARIA Authoring Practices Guide for the dialog pattern.

**WCAG:** 2.1.1 Keyboard (Level A)

**What the user hears:** A screen reader user presses Escape expecting the popup to close (as it would in every native dialog, OS alert, and common web pattern) — and nothing happens. They have no keyboard mechanism to dismiss the popup.

**Fix:** Add a `keydown` event listener on the dialog container that calls the close function when `event.key === 'Escape'`.

---

## Issue 5: Email Input Has No Programmatic Label

**Problem:** The input is `<input type="text" placeholder="Enter email">` with no associated `<label>`. Placeholder text is not a label substitute — it disappears when the user starts typing, has insufficient contrast by default, and is not consistently announced as a label by all screen readers.

**WCAG:** 1.3.1 Info and Relationships (Level A); 3.3.2 Labels or Instructions (Level A)

**What the user hears:** NVDA may announce "Enter email, edit text" using the placeholder, but once the user begins typing, the prompt is gone entirely. JAWS behavior varies. In neither case is there a persistent, programmatically associated label. The user also has no instruction that email format is required.

**Fix:** Add a visible `<label>` (preferred) or at minimum `aria-label`:
```html
<label for="newsletter-email">Email address</label>
<input type="email"
       id="newsletter-email"
       name="email"
       autocomplete="email"
       aria-describedby="email-format-hint"
       placeholder="you@example.com">
<span id="email-format-hint">
  Enter your full email address to subscribe.
</span>
```
Also change `type="text"` to `type="email"` so mobile keyboards and autocomplete behave correctly.

---

## Issue 6: "Subscribe" Button Is Not a Real Button

**Problem:** The submit control is `<div onclick="submitEmail()">Subscribe</div>`. A `<div>` has no implicit role, is not in the tab order, and does not respond to keyboard activation (Enter or Space).

**WCAG:** 4.1.2 Name, Role, Value (Level A); 2.1.1 Keyboard (Level A)

**What the user hears:** The element is either skipped entirely during Tab navigation or, if the user reaches it by other means, NVDA/JAWS announces it as unlabeled static text — not a button. Pressing Enter or Space does nothing.

**Fix:** Replace with a native `<button>`:
```html
<button type="submit">Subscribe</button>
```
Native `<button>` provides the correct implicit role, keyboard operability, and accessible name at no extra cost.

---

## Issue 7: Promotional Image Has No Alt Text

**Problem:** The large image has no `alt` attribute at all. When `alt` is absent entirely (as opposed to `alt=""`), screen readers typically announce the image's filename (e.g., "promo-banner-2024-final-v3.jpg, image"), which is meaningless or worse.

**WCAG:** 1.1.1 Non-text Content (Level A)

**What the user hears:** Something like "promo dash banner dash 2024 image" — confusing noise that adds no information and interrupts the reading flow.

**Fix:** If the image is purely decorative, use `alt=""` and `role="presentation"` to silence it:
```html
<img src="promo.jpg" alt="" role="presentation">
```
If the image conveys meaningful promotional information (e.g., "Get 20% off your first order"), that text must appear in the `alt`:
```html
<img src="promo.jpg" alt="Get 20% off your first order when you subscribe">
```

---

## Issue 8: Heading Is Not a Real Heading Element

**Problem:** `<div class="title">Don't Miss Out!!!</div>` is a styled `<div>`, not a heading element. Screen readers will not identify it as a heading; users who navigate by headings (a primary strategy for skimming page structure) will never find it.

**WCAG:** 1.3.1 Info and Relationships (Level A)

**What the user hears:** The text is read as plain paragraph content, with no announcement of it being a heading. Users navigating with H key in NVDA/JAWS will skip over it entirely.

**Fix:** Use a real heading element. Since this is inside a dialog (not the main page), `<h2>` is typically appropriate (assuming the page has an `<h1>`):
```html
<h2 id="newsletter-heading">Don't Miss Out</h2>
```
Also reconsider the double exclamation marks — some screen readers will announce each punctuation character, resulting in "Don't Miss Out exclamation exclamation exclamation."

---

## Issue 9: Body Copy Fails Color Contrast

**Problem:** Light gray text (`#cccccc`) on a white (`#ffffff`) background produces a contrast ratio of approximately **1.6:1** — far below the required minimum.

**WCAG:** 1.4.3 Contrast (Minimum), Level AA — requires **4.5:1** for normal text, **3:1** for large text.

**What the user hears:** This is primarily a low-vision issue rather than a screen reader issue, but sighted users with low vision using screen reader assist modes will be unable to read the copy. Some users combine screen readers with screen magnification and still rely on color contrast.

**Fix:** Use a minimum contrast ratio of 4.5:1 for body copy. `#767676` on white is the lightest gray that meets AA. `#595959` or darker is a safer choice.

---

## Issue 10: Validation Error Is Not Announced

**Problem:** When "Subscribe" is clicked with an empty email field, the only feedback is the input's border turning red — a purely visual change. No error text appears in the DOM, and no live region announces the problem.

**WCAG:** 3.3.1 Error Identification (Level A); 3.3.3 Error Suggestion (Level AA)

**What the user hears:** Absolutely nothing. The screen reader user clicks Subscribe, nothing happens, no error is spoken, and there is no indication of what went wrong or how to fix it.

**Fix:** Inject a descriptive error message into the DOM and associate it with the input via `aria-describedby`. Use `aria-invalid="true"` on the input to signal the error state. Announce the error via an `aria-live` region or by moving focus to the error message:
```html
<input type="email"
       id="newsletter-email"
       aria-describedby="email-error email-format-hint"
       aria-invalid="true">
<span id="email-error"
      role="alert">
  Please enter your email address.
</span>
```
`role="alert"` (which implies `aria-live="assertive"`) will cause the error to be announced immediately when it is injected into the DOM.

---

## Issue 11: Focus Styles Are Removed for All Interactive Elements

**Problem:** `outline: none` is applied to all interactive elements with no replacement focus indicator. Keyboard users — including screen reader users who rely on focus position for orientation — have no visible indication of where focus currently is.

**WCAG:** 2.4.7 Focus Visible (Level AA); 2.4.11 Focus Appearance (Level AA, WCAG 2.2)

**What the user hears/experiences:** A sighted keyboard user or low-vision user using keyboard navigation sees no focus ring anywhere and loses track of their position entirely.

**Fix:** Never use `outline: none` without providing a custom, high-contrast focus indicator as a replacement:
```css
:focus-visible {
  outline: 3px solid #005fcc;
  outline-offset: 2px;
}
```

---

## Issue 12: Popup Auto-Dismisses After 8 Seconds

**Problem:** The popup disappears automatically after 8 seconds. A screen reader user may need more than 8 seconds just to *discover* and orient themselves to the popup's contents, let alone complete the form.

**WCAG:** 2.2.1 Timing Adjustable (Level A)

**What the user hears:** Mid-interaction — possibly while the screen reader is reading the email field instructions — the popup vanishes. Focus is returned abruptly to the background page with no explanation. Any partially entered content is lost.

**Fix:** Remove the auto-dismiss behavior entirely, or provide a mechanism to pause, stop, or extend the time limit before it begins. If the popup must auto-dismiss, the time limit must be at least 20 hours (per WCAG's "essential" exception), or the user must be warned and given a way to extend it.

---

## Issue 13: No Announcement That the Popup Has Opened

**Problem:** When the popup appears 5 seconds after page load, nothing notifies a screen reader user that the page state has changed. Because focus stays on the page and there is no live region, the user is completely unaware the popup exists.

**WCAG:** 4.1.3 Status Messages (Level AA)

**What the user hears:** Nothing. The popup exists only for sighted users until and unless the screen reader user happens to navigate into it.

**Fix:** In addition to moving focus into the dialog on open (Issue 2), ensure the popup's appearance is announced. Using `role="dialog"` with `aria-modal="true"` and moving focus in will accomplish this correctly — screen readers announce the dialog's accessible name when focus enters it.

---

## Summary of Issues

| # | Issue | WCAG SC | Level |
|---|-------|---------|-------|
| 1 | No dialog role or accessible name | 4.1.2 | A |
| 2 | Focus not moved into popup on open | 2.1.1 | A |
| 3 | Close button not keyboard operable, no accessible name | 4.1.2, 2.1.1 | A |
| 4 | Escape key does not close dialog | 2.1.1 | A |
| 5 | Email input has no label | 1.3.1, 3.3.2 | A |
| 6 | Subscribe control is a `<div>`, not a `<button>` | 4.1.2, 2.1.1 | A |
| 7 | Image missing alt attribute | 1.1.1 | A |
| 8 | Heading is a `<div>`, not `<h2>` | 1.3.1 | A |
| 9 | Body text contrast ratio ~1.6:1 | 1.4.3 | AA |
| 10 | Validation error is silent | 3.3.1 | A |
| 11 | Focus indicators removed | 2.4.7 | AA |
| 12 | Auto-dismiss after 8 seconds | 2.2.1 | A |
| 13 | No announcement on popup open | 4.1.3 | AA |

---

## What's Working Well

- **The button has descriptive text.** "Subscribe" is a clear, unambiguous label — far better than "Submit" or "Click here." That intent just needs to be realized in a native `<button>` element.
- **The popup text content itself is meaningful.** "Don't Miss Out" and the email field's purpose are reasonably clear in concept — the problems are structural, not editorial.
- **The component is self-contained.** Keeping all popup content within one wrapper element is good practice and makes implementing focus trapping and `aria-modal` straightforward.

The good news is that most of these issues are fixable with correct semantic HTML — `<dialog>`, `<button>`, `<label>`, `<h2>` — before reaching for ARIA at all. Native elements do the heavy lifting for free.

---
## Reviewing as: Keyboard-Only User
*Calling model...*

---
## Review: Keyboard-Only User

### Identified Accessibility Issues

1. **Focus Management on Open/Close of Dialog (Issue with Focus Not Moving Into Popup)**
   - **Specific Problem**: When the modal popup appears, focus does not automatically move into it, preventing keyboard users from interacting with the modal's content. Additionally, there is no mechanism to return focus to the triggering element when the modal is closed.
   - **Relevant WCAG Criterion**: WCAG 2.4.3 Focus Order (Level A) and WCAG 2.4.7 Focus Visible (Level AA).
   - **Result for Keyboard User**: Keyboard users will not be able to use the email input or the subscribe button as focus is not within the modal, making it impossible to interact with the popup.
   - **Suggestion for Fix**: When the modal opens, focus should move to the email input field. Use JavaScript to set focus: `document.querySelector('input[type="text"]').focus();`. On close, return focus back to the element that triggered the popup.

2. **Lack of Visible Focus Indicators**
   - **Specific Problem**: All interactive elements have `outline: none`, meaning that there are no visible focus indicators when navigating with the keyboard. 
   - **Relevant WCAG Criterion**: WCAG 2.4.7 Focus Visible (Level AA).
   - **Result for Keyboard User**: Without a visual indicator, users cannot tell which element is currently focused. This complicates navigation and interaction.
   - **Suggestion for Fix**: Implement custom focus styles, such as a visible border or background color change on focus, using CSS: `:focus { outline: 2px solid blue; }`.

3. **No Labels Associated with Input Field**
   - **Specific Problem**: The email input field has no associated `<label>`, which means screen readers cannot convey its purpose to users.
   - **Relevant WCAG Criterion**: WCAG 4.1.2 Name, Role, Value (Level A).
   - **Result for Keyboard User**: Users relying on assistive technologies will be unable to understand what the input field is for.
   - **Suggestion for Fix**: Add a `<label for="email">Subscribe to our newsletter</label>` and associate it with the input using the input's `id`: `<input type="text" id="email" placeholder="Enter email">`.

4. **"Subscribe" Button Lacks a Button Element and Accessible Name**
   - **Specific Problem**: The "Subscribe" action is implemented using a `<div>` with an `onclick` handler instead of a proper `<button>`, which does not convey the button role and lacks an accessible name.
   - **Relevant WCAG Criterion**: WCAG 4.1.2 Name, Role, Value (Level A).
   - **Result for Keyboard User**: Screen reader users will not receive the correct context for the element as it's not recognized as a button, and they may not understand its function.
   - **Suggestion for Fix**: Replace the `<div>` with a `<button>` element: `<button onclick="submitEmail()">Subscribe</button>`. Buttons are inherently keyboard operable and contextually meaningful.

5. **Close Icon Lacks Accessible Name and Keyboard Handler**
   - **Specific Problem**: The close icon (`<span>×</span>`) has no accessible name and no keyboard functionality for closing the modal.
   - **Relevant WCAG Criterion**: WCAG 4.1.2 Name, Role, Value (Level A).
   - **Result for Keyboard User**: Users cannot identify or activate the close action using the keyboard, thus leaving the modal open or navigating away unnecessarily.
   - **Suggestion for Fix**: Use a `<button>` element for the close icon and provide an accessible name, e.g. `<button aria-label="Close modal">×</button>`. Ensure that the close button can be activated using the Enter key.

6. **Popup Visibility and Timing Issues**
   - **Specific Problem**: The modal automatically disappears after 8 seconds regardless of user action without notifying users who may need more time to read or respond.
   - **Relevant WCAG Criterion**: WCAG 2.2.1 Timing Adjustable (Level A).
   - **Result for Keyboard User**: Users may be mid-task when the modal disappears, causing frustration and missed opportunities to interact.
   - **Suggestion for Fix**: Allow users to control how long the modal remains open or remove the timeout. Also, when closing the modal automatically, provide a visual indicator or message that it has closed.

### What's Working Well
- The general structure of the popup indicates it is designed to be interactive, and there is a clear intent to collect user information (email subscription). The use of a head section suggests a promotional purpose, which is relevant for marketing efforts. By ensuring great keyboard interaction and following accessibility guidelines, users with motor impairments can have a meaningful way to engage with the popup.

---
## Reviewing as: Cognitive Disability Perspective
*Calling model...*

---
## Review: Cognitive Disability Perspective

As a user with a cognitive, learning, or attention-related disability, this "Subscribe to our Newsletter" popup presents numerous challenges that would make it difficult, frustrating, and potentially impossible to use. The design forces me to process information under time pressure, remember details, and deal with inconsistent or missing interactive cues, all of which significantly increase my cognitive load.

Here are the specific issues I've identified:

---

### Identified Accessibility Issues:

1.  **Problem:** The popup appears automatically after 5 seconds and disappears after 8 seconds, creating a very tight time limit to read, understand, and interact with it.
    *   **WCAG:** 2.2.1 Timing Adjustable (Level A)
    *   **Cognitive Impact:** For someone with a processing delay, dyslexia, or ADHD, 8 seconds is far too short to absorb the content, decide whether to subscribe, locate the input field, type an email, and click the button. I might still be reading the title when it disappears, or get distracted and lose the opportunity. This creates a high sense of pressure and leads to missed opportunities and frustration.
    *   **Fix:** The popup should persist until the user explicitly dismisses it (by clicking "close" or pressing Escape) or interacts with it. If an automatic dismissal is absolutely required (which is generally discouraged), provide an option to extend the time limit significantly or turn it off entirely.

2.  **Problem:** Body copy uses light gray text (#cccccc) on a white background, which has extremely low contrast.
    *   **WCAG:** 1.4.3 Contrast (Minimum) (Level AA)
    *   **Cognitive Impact:** This text is incredibly hard to read for anyone, but especially so for users with visual processing issues, dyslexia, or those who struggle with focus (e.g., ADHD). The effort required to decipher low-contrast text significantly increases cognitive strain and can lead to eye fatigue and abandonment of the task.
    *   **Fix:** Increase the contrast of the body copy to meet WCAG AA standards (at least 4.5:1 ratio for normal text). For instance, use a darker gray like `#333333` or `#000000` on a white background.

3.  **Problem:** The heading "Don't Miss Out!!!" is rendered as a `div` and not a semantic heading element (e.g., `<h1>`).
    *   **WCAG:** 1.3.1 Info and Relationships (Level A)
    *   **Cognitive Impact:** Screen reader users, who may also have cognitive disabilities, won't be able to easily identify this as the main title or navigate to it using heading shortcuts. While sighted users might visually understand it's a heading, the lack of semantic structure adds to cognitive load for understanding the page hierarchy, especially when using assistive technology. The multiple exclamation marks can also feel quite intense or shouty, which can be jarring.
    *   **Fix:** Use a proper semantic heading element, such as `<h1>Don't Miss Out!</h1>`. Consider reducing the number of exclamation marks to make it feel less intense.

4.  **Problem:** The email input field has placeholder text ("Enter email") but no visible, semantically associated `<label>`.
    *   **WCAG:** 3.3.2 Labels or Instructions (Level A), 1.3.1 Info and Relationships (Level A)
    *   **Cognitive Impact:** Placeholder text disappears once I start typing. If I get distracted, lose focus, or need to verify what I'm typing, the prompt is gone, forcing me to remember the field's purpose. For screen reader users, the absence of an explicit label makes it difficult or impossible to understand the field's purpose without extra navigation or guessing. This increases memory load and uncertainty.
    *   **Fix:** Add a visible and semantically associated `<label>` element for the input field, e.g., `<label for="email-field">Email address</label>`. The placeholder can still be used for additional guidance, but the label should always be present.

5.  **Problem:** The "Subscribe" button is implemented as a `div` with an `onclick` handler, not a native `<button>` element.
    *   **WCAG:** 4.1.2 Name, Role, Value (Level A), 2.1.1 Keyboard (Level A)
    *   **Cognitive Impact:** This "button" is not announced as a button by screen readers, making its purpose unclear. More critically, I cannot interact with it using a keyboard (e.g., Tab key to focus, Enter/Space key to activate). Many users with cognitive disabilities also rely on keyboard navigation, and this completely blocks interaction, leading to high frustration and an inability to complete the task.
    *   **Fix:** Use a native HTML `<button type="submit">Subscribe</button>` element. This automatically provides correct semantics, keyboard interaction, and default styling.

6.  **Problem:** The close icon (`<span>×</span>`) has no accessible name, is not keyboard focusable, and has no keyboard handler (e.g., for the Escape key).
    *   **WCAG:** 4.1.2 Name, Role, Value (Level A), 2.1.1 Keyboard (Level A)
    *   **Cognitive Impact:** Screen reader users won't know what the "x" symbol is for, or even that it's interactive. For all keyboard users (including those with motor difficulties or who find mouse use cognitively demanding), there's no way to dismiss the popup using the keyboard. This creates a trap, preventing me from returning to the main page content, causing immense frustration and increasing the cognitive effort to find a way out.
    *   **Fix:** Implement the close functionality using a semantic `<button>` element. Add `aria-label="Close newsletter popup"` to provide an accessible name. Ensure it is keyboard focusable (e.g., `tabindex="0"`) and that pressing the Escape key closes the popup.

7.  **Problem:** All interactive elements have `outline: none` and no visible focus style.
    *   **WCAG:** 2.4.7 Focus Visible (Level AA)
    *   **Cognitive Impact:** When navigating with a keyboard, I have no visual indication of which element is currently active. This makes it impossible to know where I am in the interface or which element I'm about to interact with. For someone with attention issues, this can be extremely disorienting, leading to confusion, errors, and ultimately giving up on the task.
    *   **Fix:** Restore default browser focus outlines, or provide custom, clear, and high-contrast focus indicators for all interactive elements.

8.  **Problem:** When the popup opens, keyboard focus remains on the main page content and does not move into the popup. Tab key does not move into the popup.
    *   **WCAG:** 2.4.3 Focus Order (Level A), 2.1.1 Keyboard (Level A)
    *   **Cognitive Impact:** For keyboard users, this makes the popup completely inaccessible. I cannot tab into the email field or the "Subscribe" button or the close icon. I am effectively blocked from interacting with the popup, forcing me to either abandon the task or try to use a mouse, which might not be an option or could be cognitively more demanding. This creates significant frustration and makes the popup unusable.
    *   **Fix:** When the popup appears, programmatically move keyboard focus to the first interactive element within the popup (e.g., the email input field or the close button). Ensure that focus is "trapped" within the modal, meaning pressing Tab cycles through only the elements *inside* the popup until it is dismissed.

9.  **Problem:** If "Subscribe" is clicked with an empty email field, only the input's border turns red, with no accompanying error text.
    *   **WCAG:** 3.3.1 Error Identification (Level A), 3.3.3 Error Suggestion (Level AA)
    *   **Cognitive Impact:** Relying solely on a color change for error identification is problematic for users with color vision deficiencies. More importantly, a red border doesn't explain *what* is wrong ("missing email") or *how* to fix it ("Please enter your email address"). For someone with memory or processing issues, this ambiguity adds significant cognitive load and frustration, as they're left guessing the required action.
    *   **Fix:** Provide clear, concise, and visible error messages in text next to the input field (e.g., "Please enter your email address"). Associate this error message with the input field using `aria-describedby` for screen reader users.

10. **Problem:** The large promotional image has no `alt` text.
    *   **WCAG:** 1.1.1 Non-text Content (Level A)
    *   **Cognitive Impact:** For screen reader users (who may also have cognitive disabilities), this means missing out on visual information that might be crucial to understanding the popup's purpose or message. If the image conveys important promotional value, this information is lost, creating an incomplete picture and potentially leading to confusion or a missed opportunity.
    *   **Fix:** Add appropriate `alt` text to the image that describes its content and purpose (e.g., `alt="Exclusive offers and updates on new products"`). If the image is purely decorative and its purpose is conveyed by surrounding text, an empty `alt=""` can be used.

---

### What's Working Well:

Despite the significant issues, there are a couple of points that could be built upon:

*   **Simple Task:** The primary task of entering an email address is straightforward and involves only one input field, which inherently reduces cognitive load compared to a multi-field form.
*   **Clear Goal:** The overall intent of "Subscribe to our Newsletter" is communicated visually (though the methods used are problematic). The goal of the popup is clear.

## Step 5: Synthesizer (LLM #4)

Defines the synthesizer system prompt (instructing the model to return raw JSON only) and `synthesize_reviews(component_spec, reviews)`, which sends all three reviews plus the component spec to a model via OpenRouter and returns the parsed JSON synthesis. Markdown code fences are stripped from the response before parsing.

In [ ]:
SYNTHESIS_MODEL = "anthropic/claude-sonnet-4.5"  # via OpenRouter

synthesizer_system_prompt = """You are the chair of an accessibility review panel. You have just received three independent reviews of a UI component spec from specialist reviewers — a screen reader user, a keyboard-only user, and a user with cognitive/learning/attention disabilities — along with the original component spec.

Your job is to consolidate their feedback into a single, prioritized action list the design team can act on directly.

Instructions:
- Read all three reviews and the component spec carefully.
- Deduplicate: if multiple reviewers raise the same or overlapping issue (even if worded differently), merge it into a single item and list every persona key that raised it in "raised_by".
- Assign each item a priority:
    "critical"    -> Level A violations or severe usability barriers that block task completion for affected users
    "recommended" -> Level AA violations or significant usability issues
    "enhancement" -> Level AAA criteria or best-practice improvements that go beyond minimum compliance
- Order "items" with critical issues first, then recommended, then enhancement.
- For each item include exactly these fields:
    "priority"          : "critical" | "recommended" | "enhancement"
    "change_needed"     : a specific, actionable design change (not a vague restatement of the problem)
    "raised_by"         : a list of zero or more of ["screen_reader", "keyboard", "cognitive"] - whichever persona keys raised this issue
    "wcag_criterion"    : the WCAG success criterion number and name, e.g. "1.1.1 Non-text Content"
    "conformance_level" : "A" | "AA" | "AAA"
    "rationale"         : one sentence explaining the barrier this creates for affected users
- Include a top-level "summary" field: one sentence describing the component's overall accessibility posture.

Respond with raw JSON only. Do not wrap it in a markdown code fence, and do not include any commentary, preamble, or explanation before or after the JSON object. Your entire response must be a single JSON object that can be parsed directly by json.loads()."""


def synthesize_reviews(component_spec, reviews):
    payload = {
        "component_spec":       component_spec,
        "screen_reader_review": reviews["screen_reader"],
        "keyboard_review":      reviews["keyboard"],
        "cognitive_review":     reviews["cognitive"],
    }
    messages = [
        {"role": "system", "content": synthesizer_system_prompt},
        {"role": "user",   "content": json.dumps(payload)},
    ]
    response = openrouter.chat.completions.create(model=SYNTHESIS_MODEL, messages=messages)
    content = response.choices[0].message.content.strip()
    if content.startswith("```"):
        content = content[content.index("\n") + 1:]
        if content.endswith("```"):
            content = content[:content.rfind("```")].strip()
    return json.loads(content)

## Step 6: Display the synthesized report

`display_synthesis(synthesis)` renders the synthesizer's JSON as a formatted Markdown report: a summary line with issue counts by priority, followed by a block per item showing its priority badge, WCAG criterion, change needed, which reviewers raised it, and rationale.

In [12]:
def display_synthesis(synthesis):
    items = synthesis.get("items", [])
    counts = {level: sum(1 for i in items if i["priority"] == level)
    for level in ("critical", "recommended", "enhancement")}
    md = f"""# Accessibility Panel Report

**Summary:** {synthesis['summary']}
{counts['critical']} critical · {counts['recommended']} recommended · {counts['enhancement']} enhancements

---
"""
    priority_emoji = {"critical": "🔴", "recommended": "🟡", "enhancement": "🟢"}
    for item in items:
        raised_labels = ", ".join(
            REVIEWER_PERSONAS[key] for key in item["raised_by"] if key in REVIEWER_PERSONAS
        )
        md += f"""
### {priority_emoji[item['priority']]} [{item['priority'].upper()}] WCAG {item['wcag_criterion']} (Level {item['conformance_level']})

**Change needed:** {item['change_needed']}
**Raised by:** {raised_labels}
**Rationale:** {item['rationale']}

---
"""
    display(Markdown(md))

## Optional: follow-up questions

`ask_followup(component_spec, reviews, persona, question)` sends a follow-up question to a specific reviewer as a continuation of their original conversation (system prompt → original spec → original review → new question). `followup_loop()` wraps this in an interactive prompt so the designer can ask multiple follow-ups across different personas before moving on.

In [ ]:
def ask_followup(component_spec, reviews, persona, question, max_retries=3):
    system_prompt, client, model = REVIEWER_CONFIG[persona]
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": component_spec},
        {"role": "assistant", "content": reviews[persona]},
        {"role": "user",      "content": question},
    ]
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model=model, messages=messages)
            return response.choices[0].message.content
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 20 * (attempt + 1)
            display(Markdown(f"*Rate limited — waiting {wait_seconds}s before retrying...*"))
            time.sleep(wait_seconds)


def followup_loop(component_spec, reviews):
    persona_list = ", ".join(REVIEWER_PERSONAS.keys())
    while True:
        choice = input(
            f"\nAsk a reviewer a follow-up question? "
            f"Enter a persona key ({persona_list}) or press Enter to skip: "
        ).strip().lower()
        if not choice:
            break
        if choice not in REVIEWER_PERSONAS:
            display(Markdown(f"*'{choice}' isn't one of: {persona_list}. Try again.*"))
            continue

        question = input(f"Your question for the {REVIEWER_PERSONAS[choice]}: ").strip()
        if not question:
            continue

        answer = ask_followup(component_spec, reviews, choice, question)
        display(Markdown(
            f"---\n## Follow-up — {REVIEWER_PERSONAS[choice]}\n\n**Q:** {question}\n\n{answer}"
        ))

## Optional: persist session to disk

`save_session()` writes the current `conversation_log` to a timestamped JSON file under `sessions/`, creating the directory if needed. Each run appends a new file rather than overwriting the previous one.

In [ ]:
SESSIONS_DIR = "sessions"


def save_session(filename=None):
    # Each session gets its own timestamped file in SESSIONS_DIR, so repeated
    # runs accumulate files instead of overwriting the last one.
    os.makedirs(SESSIONS_DIR, exist_ok=True)
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = os.path.join(SESSIONS_DIR, f"session_{timestamp}.json")
    data = {
        "saved_at": datetime.now().isoformat(),
        "rounds": conversation_log,
    }
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)
        print(f"Saved {len(conversation_log)} component(s) to {filename}")

In [ ]:
# Smoke-test the full chain:
component_spec = prompt_for_component()
reviews   = run_reviewers(component_spec)
synthesis = synthesize_reviews(component_spec, reviews)
display_synthesis(synthesis)


## Describe your component

Paste a description of the UI component you want reviewed. Include as much
detail as possible:

- **Component type** — e.g. modal dialog, dropdown menu, date picker, data table
- **User interaction** — how a user opens, operates, and closes it
- **Labels and text** — button labels, field labels, placeholder text, error messages
- **HTML/ARIA markup** — paste any existing code if you have it
- **Dynamic behavior** — what changes on screen when the user interacts

**Example you can imitate:**

```
A modal dialog for confirming a file deletion. It has an <h2> heading
'Delete file?', body text 'This action cannot be undone.', a red 'Delete'
button, and a 'Cancel' link. Clicking Delete closes the modal and deletes
the file. Pressing Escape should also close the modal. The modal appears
after the user clicks a trash icon on a file card.
```


## Step 7: Panel loop

`run_panel()` drives a multi-component review session: it prompts the designer to continue or quit, runs the full pipeline (`run_reviewers` → `synthesize_reviews` → `display_synthesis` → `followup_loop`) for each component, appends a record to `conversation_log`, and prints a summary of components reviewed and critical issues found when the session ends.

In [ ]:
def run_panel():
    total_critical = 0
    while True:
        choice = input(
            "\nReady to review a component? "
            "Press Enter to continue or type 'quit' to stop: "
        ).strip().lower()
        if choice == "quit":
            while True:
                save_choice = input("Would you like to save this session? (y/n): ").strip().lower()
                if save_choice == "y":
                    save_session()
                    break
                elif save_choice == "n":
                    break
            break

        component_spec = prompt_for_component()
        reviews        = run_reviewers(component_spec)
        synthesis      = synthesize_reviews(component_spec, reviews)
        display_synthesis(synthesis)
        followup_loop(component_spec, reviews)

        total_critical += sum(
            1 for item in synthesis.get("items", [])
            if item.get("priority") == "critical"
        )

        conversation_log.append({
            "component_spec": component_spec,
            "reviews":        reviews,
            "synthesis":      synthesis,
        })

    components_reviewed = len(conversation_log)
    print(f"\nSession complete. {components_reviewed} component(s) reviewed, "
        f"{total_critical} critical issue(s) identified.")


## Step 8: Run the panel

Starts an interactive review session by calling `run_panel()`. Paste a component description when prompted — the deliberately inaccessible newsletter popup spec in the previous markdown cell is a good test input to exercise all three reviewers.

In [ ]:
run_panel()


## Optional: surface recurring issues across sessions

Defines a three-stage trend pipeline over saved session files: `load_all_sessions()` globs every `sessions/*.json` file and combines their round records into one list; `tally_criteria()` counts how often each WCAG criterion was flagged and at what priority using a `Counter` and nested dict; `summarize_trends()` sends the top-N counts to Claude and displays the resulting training briefing as Markdown. The final cell runs all three stages.

In [ ]:
trends_system_prompt = """You are an accessibility program advisor helping a design team learn from their own review history.

You will be given a tally of how often each WCAG success criterion was flagged as an issue across multiple past component reviews, broken down by priority (critical / recommended / enhancement).

Your job is to turn this data into a short, actionable training briefing for the design team:
- Call out the most frequently flagged criteria by name (WCAG criterion number and title) and how many times each was flagged.
- For each one, give a one- or two-sentence plain-language explanation of the likely underlying pattern (e.g. "the team repeatedly ships custom controls without visible focus styles").
- Suggest one concrete training topic, design-system change, or checklist item per recurring issue that would help prevent it going forward.
- Keep the tone constructive — this is about systemic patterns, not blaming individual designers.

Respond in Markdown, formatted for direct display to the design team. Do not include raw JSON or code fences."""

In [ ]:
def load_all_sessions(directory=SESSIONS_DIR):
    """Combine the "rounds" from every saved session file in `directory`."""
    all_rounds = []
    json_files = glob.glob(f"{directory}/*.json")

    for f in json_files:
        with open(f, "r") as file:
            data = json.load(file)
        rounds = data.get("rounds", [])
        all_rounds.extend(rounds)

    return all_rounds

In [ ]:
def tally_criteria(all_rounds):
    """Count how often each WCAG criterion was flagged, broken down by priority."""
    overall_counts = Counter()
    counts_by_priority = {}  # e.g. {"2.4.7 Focus Visible": {"critical": 2, "recommended": 1}}

    for round_record in all_rounds:
        for item in round_record.get("synthesis", {}).get("items", []):
            criterion, priority = item.get("wcag_criterion"), item.get("priority")
            overall_counts[criterion] += 1
            inner_dict = counts_by_priority.setdefault(criterion, {})
            inner_dict[priority] = inner_dict.get(priority, 0) + 1

    return overall_counts, counts_by_priority

In [ ]:
def summarize_trends(overall_counts, counts_by_priority, top_n=10):
    """Ask Claude to turn the top N most-flagged criteria into a training briefing."""
    top_pairs = overall_counts.most_common(top_n)
    payload = [
        {
            "criterion": criterion,
            "total_count": count,
            "by_priority": counts_by_priority.get(criterion, {}),
        }
        for criterion, count in top_pairs
    ]

    messages = [
        {"role": "system", "content": trends_system_prompt},
        {"role": "user", "content": json.dumps(payload)}
    ]

    response = anthropic.chat.completions.create(
        model="claude-sonnet-4-6", messages=messages
    )

    display(Markdown(response.choices[0].message.content))

In [ ]:
all_rounds = load_all_sessions()
overall_counts, counts_by_priority = tally_criteria(all_rounds)
summarize_trends(overall_counts, counts_by_priority)